# 04 — Robustness: FFM on the raw (unsmoothed) log-volatility proxy

Re-runs the full pipeline — estimation, factor and idiosyncratic forecasts, reconstruction and model
comparison — on the raw proxy $\log(r^2 + \varepsilon_i)$, **without** B-spline smoothing,
and compares it with the smoothed pipeline of `02_forecasting.ipynb`.

Requires the shared objects saved by `01_data_eda_ffm_rolling.ipynb` and `df_cmp_smooth.pkl`
saved by `02_forecasting.ipynb` (`data/shared/`).

# Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
import os
from functools import partial

In [ ]:
# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.stattools import acf

In [ ]:
# ── Data processing & factor model ───────────────────────────────────────────

from ffm_transform import log_transform

from ffm_fx_real import run_ffm

In [ ]:
# ── EDA ───────────────────────────────────────────────────────────────────────

In [ ]:
# ── Forecasting ───────────────────────────────────────────────────────────────
from ffm_forecasting import (
    forecast_ar,
    plot_forecast_idiosyncratic,
    reconstruct_forecast,
    forecast_idio_har,
    forecast_factor_rw,
    forecast_idio_rw,
    print_model_comparison,
)

In [ ]:
# ── Stability ─────────────────────────────────────────────────────────────────

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# ── Load shared objects (saved by 01_data_eda_ffm_rolling) ────────────────────
import joblib

shared_dir = "../data/shared/"

vol_matrices_all  = joblib.load(f"{shared_dir}vol_matrices_all.pkl")
vol_matrices_2026 = joblib.load(f"{shared_dir}vol_matrices_2026.pkl")
avg_vol           = joblib.load(f"{shared_dir}avg_vol.pkl")
FX_all            = joblib.load(f"{shared_dir}FX_all.pkl")

# Data

In [ ]:
vol_matrices_2014 = {k: v.loc['2014-01-01':] for k, v in vol_matrices_all.items()}

In [ ]:
vol_sq_all  = {
    k: v[pd.to_datetime(v.index).dayofweek < 5] ** 2
    for k, v in vol_matrices_2014.items()
}
vol_log_all = log_transform(vol_sq_all)

# FFM

In [ ]:
ffm_d = run_ffm(vol_log_all,  k_max=10, min_coverage=0.7,date_start = '2014-01-01',  silent = False)

# Forecasting

In [ ]:
U_hat      = ffm_d['U_hat']
pairs_name = ffm_d['pairs']
Xi_hat     = ffm_d['Xi_hat']

## Forecasting factor 

In [ ]:
fc_rw, ci_l_rw, ci_u_rw = forecast_factor_rw(
    series  = ffm_d['U_hat'][0, :],
    horizon = [1, 2, 3, 4, 5],
    n_display = 50, 
    plot = True
)

In [ ]:
result_ar, fc_ar, ci_l_ar, ci_u_ar = forecast_ar(U_hat.T, h=5, p_max= 20,n_display=50, plot= True)

## Forecasting factor

In [ ]:
pred_rw, ci_l_rw, ci_h_rw = forecast_idio_rw(ffm_d['Xi_hat'], h=1)

plot_forecast_idiosyncratic(
    Xi_hat     = ffm_d['Xi_hat'],
    pairs_name = ffm_d['pairs'],
    h          = 1,
    forecasts  = pred_rw,
    title      = 'Idiosyncratic — Random Walk  h=1',
)

In [ ]:
from ffm_forecasting import forecast_all_assets_FAR

fc_far = forecast_all_assets_FAR(Xi_hat, ffm_d['pairs'])

plot_forecast_idiosyncratic(
    Xi_hat     = ffm_d['Xi_hat'],
    pairs_name = ffm_d['pairs'],
    h          = 1,
    forecasts  = fc_far,
    title      = 'Idiosyncratic — FAR (FPCA + VAR)  h=1',
)

In [ ]:
pred_idio_har, ci_low_har, ci_high_har = forecast_idio_har(
    Xi_hat     = ffm_d['Xi_hat'],
    pairs_name = ffm_d['pairs'],
    h          = 1,
    K          = 5,
    center_window = 252, 
    fit_window = 10
)

h=1
Xi_hat = ffm_d['Xi_hat']
pred_idio = plot_forecast_idiosyncratic(
    Xi_hat     = ffm_d['Xi_hat'],
    pairs_name = ffm_d['pairs'],
    h          = 1,
    forecasts  = pred_idio_har,
    title      = 'Idiosyncratic — HAR  h=1',
)


In [ ]:
from ffm_forecasting import plot_idio_forecast_comparison

plot_idio_forecast_comparison(
    ffm = ffm_d,   # or ffm_2025
    forecasts_dict = {
        'RW'  : (pred_rw,       ci_l_rw,     ci_h_rw),
        'FAR' : (fc_far,        None,         None),      # no CI for FAR
        'HAR' : (pred_idio_har, ci_low_har,   ci_high_har),
    },
    pairs_to_plot = ['eurusd', 'audjpy', 'eurchf', 'usdtry'],
    n_last_obs    = 10,
    h             = 1,
)

## Reconstruction

In [ ]:
target_date = pd.Timestamp('2026-01-02')

pairs       = ffm_d['pairs']
tau         = ffm_d['tau']
N           = len(pairs)

In [ ]:
h=1
X_log, X_abs = reconstruct_forecast(fc_ar, pred_idio_har, ffm_d)
print(f"h={h} : {X_abs.shape}")

In [ ]:
from ffm_forecasting import plot_reconstructed_forecast

plot_reconstructed_forecast(
    ffm  = ffm_d,
    models_dict = {
        'AR + HAR' : (fc_ar,pred_idio_har),
        'AR + FAR' : (fc_ar,fc_far),
        'AR + RW'  : (fc_ar,pred_rw),
    },
    target_date       = '2026-01-02',
    vol_ffm_2026      = None,
    vol_matrices_2026 = vol_matrices_2026,
    pairs_to_plot     = ['eurusd', 'audjpy', 'eurchf', 'usdtry'],
    h                 = 1,
)

## Model comparison (raw, no smoothing)

In [ ]:
from ffm_forecasting import run_model_comparison_raw, print_model_comparison

df_cmp_raw = run_model_comparison_raw(
    target_dates      = ['2026-01-02'],
    ffm               = ffm_d,
    vol_matrices_2026 = vol_matrices_2026,
    horizons          = [1],
    models_factor     = ['RW', 'AR'],
    models_idio       = ['RW', 'FAR', 'HAR'],
)

print_model_comparison(df_cmp_raw)

In [ ]:
from ffm_forecasting import plot_model_comparison_raw

plot_model_comparison_raw(df_cmp_raw, period='raw', h=1)

## Raw pipeline vs smoothed pipeline

Compares the FFM estimated directly on the raw data (`df_cmp_raw`, this notebook)
with the FFM estimated on the smoothed data (`df_cmp_smooth`, computed in
`02_forecasting.ipynb` and saved with joblib) — both evaluated against the
same raw observations of 2026-01-02.

In [ ]:
from ffm_forecasting import plot_comparison_raw_vs_smooth

df_cmp_smooth = joblib.load(f"{shared_dir}df_cmp_smooth.pkl")

plot_comparison_raw_vs_smooth(
    df_cmp_raw, df_cmp_smooth,
    label_raw    = 'FFM on raw data',
    label_smooth = 'FFM on smoothed data',
    h            = 1,
)